In [1]:
# Load libraries
import pandas as pd
import numpy as np

In [2]:
# Load data 
ft = pd.read_csv('data/2-processed/amplicon/feature-table_rarefied.tsv', sep='\t', header=0, index_col=0, skiprows=1)
tx = pd.read_csv('data/2-processed/amplicon/taxonomy/taxonomy.tsv', sep='\t', header=0, index_col=0)
group = pd.read_excel('data/2-processed/sample_group/analysis_dataset_group.xlsx')
group.set_index('label', inplace=True)

In [3]:
# Prevalence filter (ASV > 10%)
n = ft.shape[1]
sel = []
for id, row in ft.iterrows():
    pv = float(np.count_nonzero(row.values)) / n
    if pv > 0.1:
        sel.append(id)
print(len(sel))

ft_sel = ft.loc[sel, :]
print(ft_sel.shape)

319
(319, 483)


In [4]:
# Order feature by abundance 
tc = []
for id in sel:
    t = np.sum(ft.loc[id, :])
    tc.append([t, id])
tc.sort()
tc.reverse() 

In [5]:
# taxonomy label 
lb_cnt = {}
tx_label = {}
for e in tc: 
    fid = e[1]
    t = tx.loc[fid, 'Taxon']
    lb = t.split(';')[-1]
    if lb in lb_cnt: 
        lb_cnt[lb] += 1
    else:
        lb_cnt[lb] = 1
    
    tx_label[fid] = lb + f' ({lb_cnt[lb]})'


In [6]:
ft_labeled = ft_sel.rename(tx_label, axis=0)

In [7]:
# group
print(len(group.index))
print(ft_labeled.shape)

ci = list(set(group.index).intersection(set(ft_labeled.columns)))
print(len(ci))

484
(319, 483)
483


In [8]:
group = group.loc[ci, :]
ft_labeled = ft_labeled.loc[:, ci]
print(group.shape)
print(ft_labeled.shape)

(483, 9)
(319, 483)


In [9]:
# write the labeled feature table
ft_labeled.to_csv('data/ft_asv_tx.tsv', sep='\t')

# write the group information 
group.to_csv('data/group.tsv', sep='\t')

In [10]:
# enterotype 
cluster = pd.read_csv('data/3-results/enterotype/tsne_cluster.tsv', sep='\t', header=0, index_col=0)
cluster = cluster.loc[:, 'cluster']
cluster.to_csv('data/3-results/enterotype/enterotype.tsv', sep='\t')